# Build query-mode inputs for sv-evidence-extraction

Computes the four inputs that vary per candidate event (`region`,
`sample_ids`, `region_name`, `output_prefix`) from a chrom/start/end and
a child `IndividualID`, resolving parent sample IDs from the cohort's
pedigree file. Merges them with the stable defaults (evidence-paths
table, sample/batch map, docker image, padding) from
[`inputs/query.inputs.json`](../inputs/query.inputs.json) to produce a
ready-to-submit Terra input JSON, and optionally uploads it straight to
the workspace bucket.

Workspace-specific paths (bucket ID, cohort filenames) live in
`local_config.json` at the repo root, which is gitignored -- see
`local_config.example.json` for the expected shape. Nothing
cohort-specific is hardcoded in this notebook itself.


In [ ]:
# ==========================================
# IMPORTS
# ==========================================
import json
from pathlib import Path

import pandas as pd
from google.cloud import storage


In [ ]:
# ==========================================
# LOCAL CONFIG (workspace-specific paths)
# ==========================================
# local_config.json holds the real, workspace-specific GCS paths for this
# cohort (evidence-paths table, sample/batch map, pedigree file) and is
# gitignored -- see local_config.example.json for the expected shape.
# Keeping it out of the repo means no cohort-specific bucket ID or
# filenames end up in the public history, while this notebook still runs
# against real data locally.
#
# Assumes the notebook is run from its default location (notebooks/);
# adjust REPO_ROOT if you've moved it.
REPO_ROOT = Path.cwd().parent
LOCAL_CONFIG_PATH = REPO_ROOT / "local_config.json"

with open(LOCAL_CONFIG_PATH) as fh:
    local_config = json.load(fh)

local_config


In [ ]:
# ==========================================
# PEDIGREE LOOKUP
# ==========================================
PED_HEADER = ["FamID", "IndividualID", "FatherID", "MotherID", "Gender", "Affected"]

# PLINK-style PED files use "0" (a literal string, not a blank/NaN cell)
# as the sentinel for "no parent recorded" -- confirmed against this
# cohort's actual pedigree file, which has zero blank FatherID/MotherID
# cells but ~23k rows with FatherID == "0". A plain pd.notna() check
# would silently treat "0" as a real sample ID and try to pull evidence
# for a nonexistent sample "0".
MISSING_PARENT_SENTINELS = {"0", 0}


def load_pedigree(ped_file_uri):
    """Load a standard 6-column PED file into a DataFrame.

    Parameters
    ----------
    ped_file_uri : str
        Local path or gs:// URI to the PED file.

    Returns
    -------
    pandas.DataFrame
        Columns: FamID, IndividualID, FatherID, MotherID, Gender, Affected.
    """
    return pd.read_csv(ped_file_uri, sep="\t", names=PED_HEADER, comment="#")


def _has_parent(value):
    """True if `value` is a real parent ID, not missing/the "0" sentinel."""
    return pd.notna(value) and value not in MISSING_PARENT_SENTINELS


df_ped = load_pedigree(local_config["ped_file_uri"])
df_ped.head()


In [ ]:
# ==========================================
# RESOLVE CHILD + PARENT SAMPLE IDS
# ==========================================
def resolve_family_sample_ids(child_id, df_ped, include="both"):
    """Resolve a child's sample_ids list (itself plus requested parent(s)) from a PED table.

    Parameters
    ----------
    child_id : str
        IndividualID of the child, as it appears in df_ped["IndividualID"].
    df_ped : pandas.DataFrame
        As returned by `load_pedigree`.
    include : {"both", "father", "mother", "none"}, default "both"
        Which parent(s) to include alongside the child. "none" returns
        just the child (e.g. for a call where only the proband's own
        evidence is of interest).

    Returns
    -------
    list of str
        [child_id] plus any resolved, non-missing parent IDs, in that order.

    Raises
    ------
    ValueError
        If child_id isn't found in df_ped, since a silently-empty result
        would otherwise look like "no evidence found" downstream rather
        than "this sample isn't even in the pedigree".
    """
    matches = df_ped.loc[df_ped["IndividualID"] == child_id]
    if matches.empty:
        raise ValueError(f"'{child_id}' not found in the pedigree table.")
    row = matches.iloc[0]

    sample_ids = [child_id]
    if include in ("both", "father") and _has_parent(row["FatherID"]):
        sample_ids.append(row["FatherID"])
    if include in ("both", "mother") and _has_parent(row["MotherID"]):
        sample_ids.append(row["MotherID"])
    return sample_ids


In [ ]:
# ==========================================
# BUILD THE VARIABLE QUERY-MODE INPUTS
# ==========================================
def build_query_inputs(chrom, start, end, child_id, df_ped, include="both", name=None):
    """Compute the four variable sv-evidence-extraction query-mode inputs for one candidate event.

    The other query-mode inputs (evidence_paths_tsv, sample_batch_map_tsv,
    docker, padding, etc.) are stable across events and come from the
    checked-in inputs/query.inputs.json template plus local_config.json,
    not from this function -- see `write_query_inputs_json`.

    Parameters
    ----------
    chrom : str
    start, end : int
        1-based inclusive core event coordinates (padding is applied by
        the WDL/CLI itself, not here).
    child_id : str
        IndividualID of the proband, used to resolve parents via df_ped.
    df_ped : pandas.DataFrame
        As returned by `load_pedigree`.
    include : {"both", "father", "mother", "none"}, default "both"
        Which parent(s) to pull evidence for alongside the child.
    name : str, optional
        Human-readable label for this event; auto-generated from
        chrom/start/end/child_id if not given.

    Returns
    -------
    dict
        Keys "region", "sample_ids", "region_name", "output_prefix" --
        exactly the TBD fields in inputs/query.inputs.json.
    """
    sample_ids = resolve_family_sample_ids(child_id, df_ped, include=include)
    region_name = name or f"{chrom}_{start}_{end}_{child_id}"

    return {
        "region": f"{chrom}:{start}-{end}",
        "sample_ids": ",".join(sample_ids),
        "region_name": region_name,
        "output_prefix": region_name,
    }


In [ ]:
# ==========================================
# MERGE INTO THE QUERY-MODE INPUT TEMPLATE
# ==========================================
def write_query_inputs_json(variable_inputs, local_config, template_path, out_path):
    """Merge computed variable inputs with the stable template/local-config defaults, and write the result.

    Parameters
    ----------
    variable_inputs : dict
        As returned by `build_query_inputs` -- region, sample_ids,
        region_name, output_prefix.
    local_config : dict
        As loaded from local_config.json -- supplies evidence_paths_tsv
        and sample_batch_map_tsv.
    template_path : str or pathlib.Path
        Path to the checked-in inputs/query.inputs.json (structural
        template: key names, padding defaults, docker image).
    out_path : str or pathlib.Path
        Where to write the merged, ready-to-submit JSON.

    Returns
    -------
    dict
        The merged input dictionary that was written to `out_path`.
    """
    with open(template_path) as fh:
        merged = json.load(fh)

    merged["SVEvidenceExtraction.evidence_paths_tsv"] = local_config["evidence_paths_tsv"]
    merged["SVEvidenceExtraction.sample_batch_map_tsv"] = local_config["sample_batch_map_tsv"]
    merged["SVEvidenceExtraction.region"] = variable_inputs["region"]
    merged["SVEvidenceExtraction.sample_ids"] = variable_inputs["sample_ids"]
    merged["SVEvidenceExtraction.region_name"] = variable_inputs["region_name"]
    merged["SVEvidenceExtraction.output_prefix"] = variable_inputs["output_prefix"]

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as fh:
        json.dump(merged, fh, indent=2)

    return merged


In [ ]:
# ==========================================
# UPLOAD TO THE WORKSPACE BUCKET (OPTIONAL)
# ==========================================
def upload_to_workspace_bucket(local_path, workspace_bucket, upload_prefix, gcp_billing_project):
    """Upload a local file to the Terra workspace bucket, so Terra can read it directly.

    Falls back gracefully (prints a warning, returns None) rather than
    raising, since this notebook should still be usable purely locally
    -- e.g. on a machine without write access to this specific bucket --
    with the caller uploading the resulting file by hand instead.

    Parameters
    ----------
    local_path : str or pathlib.Path
        File to upload.
    workspace_bucket : str
        GCS bucket name (no gs:// prefix), from local_config.json.
    upload_prefix : str
        Folder within the bucket to upload into, from local_config.json.
    gcp_billing_project : str
        GCP project to bill the API call's quota to -- any project you
        have usage rights on works; it does not need to own the bucket.

    Returns
    -------
    str or None
        The resulting gs:// URI, or None if the upload failed.
    """
    local_path = Path(local_path)
    blob_name = f"{upload_prefix.rstrip('/')}/{local_path.name}"
    try:
        client = storage.Client(project=gcp_billing_project)
        bucket = client.bucket(workspace_bucket)
        bucket.blob(blob_name).upload_from_filename(str(local_path))
    except Exception as exc:
        print(f"NOTE: upload failed ({exc}); {local_path} was still written locally -- upload it by hand instead.")
        return None

    gs_uri = f"gs://{workspace_bucket}/{blob_name}"
    print(f"Uploaded to {gs_uri}")
    return gs_uri


## Example: one candidate event

Replace the coordinates and `child_id` below with a real candidate --
this example uses placeholder values only.


In [ ]:
# ==========================================
# EXAMPLE: ONE CANDIDATE EVENT
# ==========================================
example_inputs = build_query_inputs(
    chrom="chr1",
    start=1000000,
    end=1005000,
    child_id="REPLACE_WITH_A_REAL_CHILD_ID",
    df_ped=df_ped,
    include="both",
)
example_inputs


In [ ]:
# ==========================================
# WRITE + (OPTIONALLY) UPLOAD THE READY-TO-SUBMIT JSON
# ==========================================
out_path = REPO_ROOT / "notebooks" / "generated" / f"{example_inputs['region_name']}.query.inputs.json"
merged = write_query_inputs_json(
    variable_inputs=example_inputs,
    local_config=local_config,
    template_path=REPO_ROOT / "inputs" / "query.inputs.json",
    out_path=out_path,
)
print(f"Wrote {out_path}")

upload_to_workspace_bucket(
    local_path=out_path,
    workspace_bucket=local_config["workspace_bucket"],
    upload_prefix=local_config["upload_prefix"],
    gcp_billing_project=local_config["gcp_billing_project"],
)
